**UNSUPERVISED DEEP LEARNING ASSIGNMENT 2**

PART C: Comparative Analysis

GROUP 78

GROUP MEMBERS:

---

| SR NO | STUDENT NAME | ENROLLMENT NUMBER | CONTRIBUTION |
|---|---|---|---|
| 1 | RITU SAXENA | 2025ab05142 | 100% |
| 2 | DHUMAL ATHARVA SANJAY | 2025ab05230 | 100% |
| 3 | AKHILESH DEEPAK JICHKAR | 2025aa05613 | 100% |
| 4 | EMMADISHETTY SRI ADITYA VARMA | 2025ab05080 | 100% |
| 5 | SHELKE RAHUL PANDURANG PURNIMA | 2025aa05484 | 100% |


## Part C: Comparative Analysis

Produces the required deliverable: **100 visual examples** for beta-VAE (beta=1), VQ-VAE (K=256), WGAN and
WGAN-GP, a comparison of evaluation scores across all four, and a discussion of reconstruction error,
sampling quality and speed - exported as `GROUP_78_PART_C.pdf`.

**How to run this notebook**

1. Open this notebook in the **same Colab environment used for Part A** (`GROUP_78_PART_A.ipynb`), so its
   Google Drive checkpoint folder (`udl_assignment2_part_a_checkpoints`) is reachable. Two supported paths:
   - **Same live runtime as Part A** (you just ran `Runtime > Run all` on Part A and haven't disconnected):
     just run this notebook's cells - it will detect and reuse `vae_models`, `vqvae_models`,
     `pixelcnn_models` already in memory, no reloading needed.
   - **Fresh runtime**: this notebook mounts Drive itself and loads the three saved checkpoints
     (`bvae_beta1.pt`, `vqvae_K256.pt`, `pixelcnn_K256.pt`) from `CKPT_DIR` below. Update `CKPT_DIR` if you
     used a different Drive folder name in Part A.
2. Before running the WGAN/WGAN-GP cell, upload the two files from this delivery's `part_c_assets/` folder
   (`wgan_100_samples.jpg`, `wgangp_100_samples.jpg`) into the Colab session's file browser (left sidebar ->
   folder icon -> upload). These are the actual 100-sample grids already generated and saved inside
   `GROUP_78_PART_B.ipynb` - Part B's GAN checkpoints weren't retained locally, so rather than fabricate new
   samples this notebook reuses the real ones Part B already produced.
3. `Runtime > Run all`. The final cell writes `part_c_outputs/GROUP_78_PART_C.pdf`.

**Why this design:** Part A's own notebook (see its final "Summary" cell) already intended to save 100-sample
grids for beta=1 / K=256 to disk, but (a) `N_SAMPLES` was set to 64, not 100, when generating VQ-VAE+PixelCNN
samples, and (b) neither grid was ever displayed/embedded in the notebook, only written to a local Colab
path that isn't part of this delivery. This notebook regenerates both correctly at n=100 and times the
generation, which also gives real numbers for the "speed" comparison the brief asks for.


In [ ]:
import os, io, sys, time, json, textwrap
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
import pandas as pd

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print("Drive mount skipped/failed:", e)

# Must match CKPT_DIR used in GROUP_78_PART_A.ipynb - change if you used a different Drive folder.
CKPT_DIR = "/content/drive/MyDrive/udl_assignment2_part_a_checkpoints"
OUT_DIR = "part_c_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


## 1. Model architectures (copied verbatim from Part A, needed to load its checkpoints)

In [ ]:
# --- beta-VAE (identical to GROUP_78_PART_A.ipynb) ---
VAE_LATENT = 128

class VAEEncoder(nn.Module):
    def __init__(self, latent_dim=VAE_LATENT):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
        self.fc_mu = nn.Linear(256 * 2 * 2, latent_dim)
        self.fc_logvar = nn.Linear(256 * 2 * 2, latent_dim)

    def forward(self, x):
        h = self.conv(x).flatten(1)
        return self.fc_mu(h), self.fc_logvar(h)

class VAEDecoder(nn.Module):
    def __init__(self, latent_dim=VAE_LATENT):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 256 * 2 * 2)
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, 4, 2, 1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 32, 4, 2, 1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 3, 4, 2, 1), nn.Sigmoid(),
        )

    def forward(self, z):
        h = self.fc(z).view(-1, 256, 2, 2)
        return self.deconv(h)

class BetaVAE(nn.Module):
    def __init__(self, latent_dim=VAE_LATENT):
        super().__init__()
        self.encoder = VAEEncoder(latent_dim)
        self.decoder = VAEDecoder(latent_dim)
        self.latent_dim = latent_dim

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar

    @torch.no_grad()
    def sample(self, n, device):
        z = torch.randn(n, self.latent_dim, device=device)
        return self.decoder(z)


# --- VQ-VAE (identical to GROUP_78_PART_A.ipynb) ---
VQ_HIDDEN, VQ_RES_H, VQ_NUM_RES, VQ_EMBED_DIM, VQ_COMMIT = 128, 32, 2, 64, 0.25

class ResBlock(nn.Module):
    def __init__(self, cin, ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReLU(True), nn.Conv2d(cin, ch, 3, 1, 1),
            nn.ReLU(True), nn.Conv2d(ch, cin, 1, 1, 0),
        )
    def forward(self, x):
        return x + self.block(x)

class VQVAEEncoder(nn.Module):
    def __init__(self, cin=3, h=VQ_HIDDEN, res_h=VQ_RES_H, n_res=VQ_NUM_RES, out=VQ_EMBED_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, h // 2, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(h // 2, h, 4, 2, 1), nn.ReLU(True),
            nn.Conv2d(h, h, 3, 1, 1),
            *[ResBlock(h, res_h) for _ in range(n_res)],
            nn.ReLU(True),
            nn.Conv2d(h, out, 1, 1, 0),
        )
    def forward(self, x):
        return self.net(x)

class VQVAEDecoder(nn.Module):
    def __init__(self, cin=VQ_EMBED_DIM, h=VQ_HIDDEN, res_h=VQ_RES_H, n_res=VQ_NUM_RES, out=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(cin, h, 3, 1, 1),
            *[ResBlock(h, res_h) for _ in range(n_res)],
            nn.ReLU(True),
            nn.ConvTranspose2d(h, h // 2, 4, 2, 1), nn.ReLU(True),
            nn.ConvTranspose2d(h // 2, out, 4, 2, 1), nn.Sigmoid(),
        )
    def forward(self, z_q):
        return self.net(z_q)

class VectorQuantizer(nn.Module):
    def __init__(self, K, D, commit=VQ_COMMIT):
        super().__init__()
        self.K, self.D, self.commit = K, D, commit
        self.embedding = nn.Embedding(K, D)
        self.embedding.weight.data.uniform_(-1 / K, 1 / K)

    def forward(self, z_e):
        z = z_e.permute(0, 2, 3, 1).contiguous()
        flat = z.reshape(-1, self.D)
        dist = (flat.pow(2).sum(1, keepdim=True)
                - 2 * flat @ self.embedding.weight.t()
                + self.embedding.weight.pow(2).sum(1))
        idx = dist.argmin(1)
        z_q = self.embedding(idx).view(z.shape)
        z_q_st = z + (z_q - z).detach()
        z_q_st = z_q_st.permute(0, 3, 1, 2).contiguous()
        return z_q_st, None, None, idx.view(z_e.size(0), z_e.size(2), z_e.size(3))

class VQVAE(nn.Module):
    def __init__(self, K, embed_dim=VQ_EMBED_DIM):
        super().__init__()
        self.encoder = VQVAEEncoder(out=embed_dim)
        self.vq = VectorQuantizer(K, embed_dim)
        self.decoder = VQVAEDecoder(cin=embed_dim)

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, perplexity, indices = self.vq(z_e)
        return self.decoder(z_q), vq_loss, perplexity, indices


# --- PixelCNN prior (identical to GROUP_78_PART_A.ipynb) ---
PCNN_CHANNELS, PCNN_LAYERS, PCNN_KERNEL, LATENT_HW = 128, 6, 5, 8

class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert mask_type in ("A", "B")
        self.register_buffer("mask", self.weight.data.clone())
        _, _, kh, kw = self.weight.shape
        yc, xc = kh // 2, kw // 2
        self.mask.fill_(1)
        self.mask[:, :, yc, xc + (1 if mask_type == "B" else 0):] = 0
        self.mask[:, :, yc + 1:, :] = 0

    def forward(self, x):
        self.weight.data *= self.mask
        return super().forward(x)

class PixelCNN(nn.Module):
    def __init__(self, K, hidden=PCNN_CHANNELS, n_layers=PCNN_LAYERS, kernel_size=PCNN_KERNEL):
        super().__init__()
        self.K = K
        pad = kernel_size // 2
        self.embed = nn.Embedding(K, hidden)
        layers = [MaskedConv2d("A", hidden, hidden, kernel_size, padding=pad), nn.ReLU(inplace=True)]
        for _ in range(n_layers - 1):
            layers += [MaskedConv2d("B", hidden, hidden, kernel_size, padding=pad), nn.ReLU(inplace=True)]
        self.net = nn.Sequential(*layers)
        self.out = nn.Conv2d(hidden, K, 1)

    def forward(self, x):
        h = self.embed(x).permute(0, 3, 1, 2)
        h = self.net(h)
        return self.out(h)

@torch.no_grad()
def sample_pcnn(model, n, hw=LATENT_HW, temperature=1.0):
    model.eval()
    grid = torch.zeros(n, hw, hw, dtype=torch.long, device=device)
    for i in range(hw):
        for j in range(hw):
            logits = model(grid)
            probs = F.softmax(logits[:, :, i, j] / temperature, dim=1)
            grid[:, i, j] = torch.multinomial(probs, 1).squeeze(1)
    return grid

print("Architectures defined.")


## 2. Load the trained beta=1 VAE, K=256 VQ-VAE and K=256 PixelCNN prior

In [ ]:
if "vae_models" not in globals(): vae_models = {}
if "vqvae_models" not in globals(): vqvae_models = {}
if "pixelcnn_models" not in globals(): pixelcnn_models = {}

if 1 in vae_models:
    print("Reusing beta-VAE (beta=1) already in memory from Part A.")
else:
    m = BetaVAE(VAE_LATENT).to(device)
    ckpt = torch.load(f"{CKPT_DIR}/bvae_beta1.pt", map_location=device)
    m.load_state_dict(ckpt["model_state"]); m.eval()
    vae_models[1] = m
    print("Loaded beta-VAE (beta=1) from", f"{CKPT_DIR}/bvae_beta1.pt")

if 256 in vqvae_models:
    print("Reusing VQ-VAE (K=256) already in memory from Part A.")
else:
    m = VQVAE(256).to(device)
    ckpt = torch.load(f"{CKPT_DIR}/vqvae_K256.pt", map_location=device)
    m.load_state_dict(ckpt["model_state"]); m.eval()
    vqvae_models[256] = m
    print("Loaded VQ-VAE (K=256) from", f"{CKPT_DIR}/vqvae_K256.pt")

if 256 in pixelcnn_models:
    print("Reusing PixelCNN prior (K=256) already in memory from Part A.")
else:
    m = PixelCNN(256).to(device)
    ckpt = torch.load(f"{CKPT_DIR}/pixelcnn_K256.pt", map_location=device)
    m.load_state_dict(ckpt["model_state"]); m.eval()
    pixelcnn_models[256] = m
    print("Loaded PixelCNN prior (K=256) from", f"{CKPT_DIR}/pixelcnn_K256.pt")


## 3. Generate 100 fresh samples per model, timed

In [ ]:
torch.manual_seed(SEED)
t0 = time.time()
with torch.no_grad():
    vae_100 = vae_models[1].eval().sample(100, device).cpu()
vae_gen_time = time.time() - t0
print(f"beta-VAE (beta=1): generated 100 samples in {vae_gen_time:.3f}s ({100/vae_gen_time:.1f} img/s)")

vae_grid_t = vutils.make_grid(vae_100.clamp(0, 1), nrow=10, padding=1)
vutils.save_image(vae_grid_t, f"{OUT_DIR}/vae_beta1_100_samples.png")
vae_grid_np = vae_grid_t.permute(1, 2, 0).numpy()

plt.figure(figsize=(8, 8)); plt.axis("off")
plt.title("beta-VAE (beta=1) - 100 samples from prior")
plt.imshow(vae_grid_np); plt.show()


In [ ]:
torch.manual_seed(SEED)
t0 = time.time()
with torch.no_grad():
    codes = sample_pcnn(pixelcnn_models[256], 100)
    z_q = vqvae_models[256].vq.embedding(codes).permute(0, 3, 1, 2).contiguous()
    vqvae_100 = vqvae_models[256].decoder(z_q).cpu()
vqvae_gen_time = time.time() - t0
print(f"VQ-VAE (K=256) + PixelCNN: generated 100 samples in {vqvae_gen_time:.3f}s ({100/vqvae_gen_time:.1f} img/s)")
print(f"  -> {vqvae_gen_time / vae_gen_time:.1f}x slower than the VAE's single-pass sampling, "
      f"due to the {LATENT_HW*LATENT_HW}-step autoregressive PixelCNN prior")

vqvae_grid_t = vutils.make_grid(vqvae_100.clamp(0, 1), nrow=10, padding=1)
vutils.save_image(vqvae_grid_t, f"{OUT_DIR}/vqvae_k256_100_samples.png")
vqvae_grid_np = vqvae_grid_t.permute(1, 2, 0).numpy()

plt.figure(figsize=(8, 8)); plt.axis("off")
plt.title("VQ-VAE (K=256) + PixelCNN - 100 samples")
plt.imshow(vqvae_grid_np); plt.show()


## 4. WGAN / WGAN-GP samples (reused from Part B)

Part B's generator checkpoints (`generator_wgan.pt`, `generator_wgangp.pt`) were saved to the local Colab
disk, not Drive, so they aren't available in this session. Rather than retrain WGAN/WGAN-GP from scratch
(which Part C isn't asking for), this cell reuses the exact 100-sample grids Part B already generated and
displayed (`GROUP_78_PART_B.ipynb`, "Generate random samples" sections) - upload the two files from
`part_c_assets/` first (see instructions at the top of this notebook).

In [ ]:
wgan_path = "wgan_100_samples.jpg"
wgangp_path = "wgangp_100_samples.jpg"

assert os.path.exists(wgan_path), (
    f"'{wgan_path}' not found - upload part_c_assets/{wgan_path} to the Colab file browser first."
)
assert os.path.exists(wgangp_path), (
    f"'{wgangp_path}' not found - upload part_c_assets/{wgangp_path} to the Colab file browser first."
)

wgan_img = np.array(Image.open(wgan_path).convert("RGB"))
wgangp_img = np.array(Image.open(wgangp_path).convert("RGB"))

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(wgan_img); axes[0].axis("off"); axes[0].set_title("WGAN - 100 samples (from Part B)")
axes[1].imshow(wgangp_img); axes[1].axis("off"); axes[1].set_title("WGAN-GP - 100 samples (from Part B)")
plt.show()


## 5. Evaluation-score comparison

PSNR/FID for beta-VAE and VQ-VAE are the real values reported in `GROUP_78_PART_A.ipynb` (15 epochs,
15,000-image train subset). FID for WGAN/WGAN-GP is the real value reported in `GROUP_78_PART_B.ipynb`
(full 50,000-image train set). Sampling speed for beta-VAE and VQ-VAE is measured live above; WGAN/WGAN-GP
speed is not re-benchmarked here since their checkpoints aren't available in this session - see the
discussion below for why that's architecturally predictable anyway.

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "beta-VAE (beta=1)",
        "Recon PSNR (dB)": 17.52,
        "Recon FID": None,
        "Sample FID": 363.83,
        "Sampling speed / 100 imgs": f"{vae_gen_time:.2f}s ({100/vae_gen_time:.1f} img/s)",
    },
    {
        "Model": "VQ-VAE (K=256) + PixelCNN",
        "Recon PSNR (dB)": 20.45,
        "Recon FID": 231.63,
        "Sample FID": 325.71,
        "Sampling speed / 100 imgs": f"{vqvae_gen_time:.2f}s ({100/vqvae_gen_time:.1f} img/s)",
    },
    {
        "Model": "WGAN",
        "Recon PSNR (dB)": None,
        "Recon FID": None,
        "Sample FID": 98.62,
        "Sampling speed / 100 imgs": "not re-benchmarked (see discussion) - single forward pass, no autoregression",
    },
    {
        "Model": "WGAN-GP",
        "Recon PSNR (dB)": None,
        "Recon FID": None,
        "Sample FID": 80.79,
        "Sampling speed / 100 imgs": "not re-benchmarked (see discussion) - single forward pass, no autoregression",
    },
])
comparison


In [ ]:
discussion = f"""
RECONSTRUCTION ERROR
Only the two autoencoder-based models (beta-VAE and VQ-VAE) reconstruct their inputs, so reconstruction
error is compared between them via test-set PSNR. VQ-VAE (K=256) reconstructs cleaner than beta-VAE
(beta=1): 20.45 dB vs 17.52 dB. This matches the architectural difference - the discrete codebook lets
VQ-VAE encode a much higher-capacity representation of each 8x8 grid of local patches than the single
128-dim Gaussian bottleneck used by the VAE, and VQ-VAE has no KL term pulling its latents toward a fixed
prior, so it can spend its full capacity on fidelity rather than trading it off against regularization.
WGAN and WGAN-GP have no encoder - they only map noise to images - so PSNR is not defined for them.

SAMPLING / PERCEPTUAL QUALITY
Ranked by FID (lower is better): WGAN-GP (80.79) < WGAN (98.62) << VQ-VAE+PixelCNN (325.71) <
beta-VAE (363.83). The GANs produce visibly more realistic CIFAR-10-like samples in the grids above -
WGAN-GP shows more coherent object silhouettes and fewer flat/noisy patches than WGAN. Both beta-VAE and
VQ-VAE+PixelCNN samples are noticeably blurrier and less structured, consistent with the known gap between
adversarial and likelihood-based generators: the VAE's pixel-wise MSE objective encourages averaging over
plausible outputs (blur), and PixelCNN sampling on top of VQ-VAE compounds a second layer of approximation
error on top of the VQ-VAE decoder's own reconstruction error - which is why its sample FID (325.71) is far
worse than its reconstruction FID (231.63). Note both Part A models were trained for only 15 epochs on a
15,000-image subset (a deliberate compute-budget tradeoff documented in Part A), so their absolute FID
values understate what these architectures can reach with the full dataset and a longer schedule; the
relative ranking (GANs > VQ-VAE > VAE) would be expected to hold regardless, though the gap would likely
narrow.

SPEED
Sampling speed differs by architecture, not just training budget. beta-VAE and both WGAN variants generate
an image with a single decoder/generator forward pass, so 100 samples cost one batched forward pass -
measured at {vae_gen_time:.2f}s for the VAE here. WGAN/WGAN-GP were not re-benchmarked in this notebook
since their checkpoints were not retained locally between Part B and Part C, but they share the same
single-pass architecture as the VAE decoder, so a comparable wall-clock time is expected - certainly nowhere
near VQ-VAE+PixelCNN's cost. VQ-VAE+PixelCNN is structurally much slower to sample from: its PixelCNN prior
is autoregressive over the {LATENT_HW}x{LATENT_HW}={LATENT_HW*LATENT_HW}-cell latent grid, requiring
{LATENT_HW*LATENT_HW} sequential forward passes through the prior network (one per latent position) before
a single batched decode - measured at {vqvae_gen_time:.2f}s for 100 images here, {vqvae_gen_time/vae_gen_time:.0f}x
slower than the VAE on the same GPU and batch size. This sequential-sampling cost is a well-known drawback
of PixelCNN-style priors and is the main practical downside of the VQ-VAE+PixelCNN pipeline relative to
single-pass generators like VAEs and GANs.

OVERALL
Taking reconstruction error, sample realism (FID) and sampling speed together: WGAN-GP is the strongest
generator in this comparison - best FID and fast single-pass sampling - at the cost of providing no
encoder/reconstruction pathway. VQ-VAE gives the best reconstruction fidelity and a genuine encoder (useful
when compression/reconstruction matters, not just generation), but its PixelCNN-prior sampling is both
slower and lower-fidelity than the GANs' output. beta-VAE is the weakest performer on both PSNR and FID
among the four at this training budget, though it remains the simplest and cheapest model to train and
sample from among the two non-adversarial methods.
""".strip()

print(discussion)


## 6. Export everything to the required PDF

In [ ]:
pdf_path = f"{OUT_DIR}/GROUP_78_PART_C.pdf"

def add_text_page(pdf, title, body, lines_per_page=48, fontsize=8.7):
    paragraphs = [p for p in body.split("\n\n") if p.strip()]
    wrapped_lines = []
    for p in paragraphs:
        head, _, rest = p.strip().partition("\n")
        wrapped_lines.append(head)
        wrapped_lines.extend(textwrap.wrap(rest.replace("\n", " ").strip(), 100))
        wrapped_lines.append("")

    for start in range(0, len(wrapped_lines), lines_per_page):
        chunk = wrapped_lines[start:start + lines_per_page]
        fig = plt.figure(figsize=(8.27, 11.69))
        plt.axis("off")
        if start == 0:
            fig.text(0.07, 0.96, title, fontsize=14, weight="bold", va="top")
            y0 = 0.91
        else:
            y0 = 0.96
        fig.text(0.07, y0, "\n".join(chunk), fontsize=fontsize, va="top", family="monospace")
        pdf.savefig(fig)
        plt.close(fig)

with PdfPages(pdf_path) as pdf:
    # --- Title page ---
    fig = plt.figure(figsize=(8.27, 11.69))
    plt.axis("off")
    fig.text(0.5, 0.90, "UNSUPERVISED DEEP LEARNING - ASSIGNMENT 2", ha="center", fontsize=15, weight="bold")
    fig.text(0.5, 0.865, "PART C: Comparative Analysis", ha="center", fontsize=13)
    fig.text(0.5, 0.83, "GROUP 78", ha="center", fontsize=11)
    members = [
        ["1", "RITU SAXENA", "2025ab05142"],
        ["2", "DHUMAL ATHARVA SANJAY", "2025ab05230"],
        ["3", "AKHILESH DEEPAK JICHKAR", "2025aa05613"],
        ["4", "EMMADISHETTY SRI ADITYA VARMA", "2025ab05080"],
        ["5", "SHELKE RAHUL PANDURANG PURNIMA", "2025aa05484"],
    ]
    ax = fig.add_axes([0.15, 0.55, 0.7, 0.2])
    ax.axis("off")
    tbl = ax.table(cellText=members, colLabels=["Sr No", "Student Name", "Enrollment No"],
                    loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.6)
    fig.text(0.5, 0.40, "Includes: 100 generated samples for beta-VAE (beta=1), VQ-VAE (K=256),\n"
                        "WGAN and WGAN-GP; evaluation-score comparison; and a discussion of\n"
                        "reconstruction error, sampling quality and speed.",
             ha="center", fontsize=9.5)
    pdf.savefig(fig); plt.close(fig)

    # --- Comparison table page ---
    fig, ax = plt.subplots(figsize=(11.69, 8.27))
    ax.axis("off")
    ax.set_title("Evaluation score comparison", fontsize=15, pad=20)
    tbl = ax.table(cellText=comparison.fillna("-").values.tolist(),
                    colLabels=comparison.columns.tolist(),
                    loc="center", cellLoc="left")
    tbl.auto_set_font_size(False); tbl.set_fontsize(8.5); tbl.scale(1, 2.4)
    tbl.auto_set_column_width(col=list(range(len(comparison.columns))))
    pdf.savefig(fig); plt.close(fig)

    # --- Image pages ---
    image_pages = [
        ("beta-VAE (beta=1) - 100 generated samples", vae_grid_np,
         "Recon PSNR 17.52 dB  |  Sample FID 363.83"),
        ("VQ-VAE (K=256) + PixelCNN - 100 generated samples", vqvae_grid_np,
         "Recon PSNR 20.45 dB  |  Recon FID 231.63  |  Sample FID 325.71"),
        ("WGAN - 100 generated samples", wgan_img, "Sample FID 98.62"),
        ("WGAN-GP - 100 generated samples", wgangp_img, "Sample FID 80.79"),
    ]
    for title, img, caption in image_pages:
        fig = plt.figure(figsize=(8.27, 9.8))
        plt.imshow(img); plt.axis("off")
        plt.title(title, fontsize=13)
        fig.text(0.5, 0.045, caption, ha="center", fontsize=10)
        pdf.savefig(fig); plt.close(fig)

    # --- Discussion pages ---
    add_text_page(pdf, "Comparative discussion", discussion)

print("Saved:", pdf_path)
print("Download it from the Colab file browser, or run:")
print(f"  from google.colab import files; files.download(\'{pdf_path}\')")


### Notes for the write-up

- All numbers and images above come from the actual executed runs in `GROUP_78_PART_A.ipynb` and
  `GROUP_78_PART_B.ipynb` (or, for the VAE/VQ-VAE sample grids and their sampling speed, from re-running
  the already-trained checkpoints in this notebook) - nothing here is fabricated or estimated.
- If you retrain any of the four models with a larger epoch budget or the full 50,000-image CIFAR-10 train
  set, re-run this notebook to refresh the PDF with updated numbers/images before submitting.
